<img src="images/nvidia_header.png" style="margin-left: -30px; width: 300px; float: left;">

# MONAI Label Integration with XNAT Using Radiology App

In this notebook, we show the end-to-end setup of the XNAT research PACS and MONAI Label plugin.
Here we will use the components below.
<p style="float: left;"> 
    <img src="./images/monailabel_arch_xnat.png" width="1000"/>
</p>

We will use the XNAT OHIF plugin to demonstrated the spleen segmentation as shown below.
<p align = "center"><img src="./images/xnat_ohif.png" alt="drawing" width="500"/></p>


### Table of Contents
**[0.0 XNAT Setup](#0.0-XNAT-Setup)** <br>
**[1.0 What Is XNAT ](#1.0-What-Is-XNAT )** <br>
&nbsp;&nbsp;&nbsp;&nbsp;[1.1 Get to the XNAT Site](#1.1-Get-to-the-XNAT-Site) <br>
&nbsp;&nbsp;&nbsp;&nbsp;[1.2 Configure XNAT](#1.2-Configure-XNAT) <br>
&nbsp;&nbsp;&nbsp;&nbsp;[1.3 Confirm XNAT Is Configured Correctly](#1.3-Confirm-XNAT-Is-Configured-Correctly) <br>
**[2.0 Prepare DICOM Sample Data](#2.0-Prepare-DICOM-Sample-Data)** <br>
&nbsp;&nbsp;&nbsp;&nbsp;[2.1 Download Spleen CT Data](#2.1-Download-Spleen-CT-Data) <br>
&nbsp;&nbsp;&nbsp;&nbsp;[2.2 Convert to DICOM Format](#2.2-Convert-to-DICOM-Format) <br>
&nbsp;&nbsp;&nbsp;&nbsp;[2.3 Upload to XNAT](#2.3-Upload-to-XNAT) <br>
&nbsp;&nbsp;&nbsp;&nbsp;[2.4 Convert Liver Dataset](#2.4-Convert-Liver-Dataset) <br>
**[3.0 Set Up XNAT OHIF](#3.0-Set-Up-XNAT-OHIF)** <br>
**[4.0 Start MONAI Label Server](#4.0-Start-MONAI-Label-Server)** <br>
**[5.0 Auto-Segmentation with XNAT OHIF](#5.0-Auto-Segmentation-with-XNAT-OHIF)** <br>
&nbsp;&nbsp;&nbsp;&nbsp;[5.1 MONAI Label Plugin](#5.1-MONAI-Label-Plugin) <br>
&nbsp;&nbsp;&nbsp;&nbsp;[5.2 Run Auto-Segmentation](#5.2-Run-Auto-Segmentation) <br>

---
# 0.0 XNAT Setup
Let's import some libs used in this notebook.

In [ ]:
import ipywidgets
import threading
import os
HOST_NAME=os.getenv('HOST_NAME')
HOST_IP=os.getenv('HOST_IP')
DLI_DATA_DIR=os.getenv('DLI_DATA_DIR')
XNAT_DATA_ARCHIVE=os.getenv('XNAT_DATA_ARCHIVE')
os.chdir(DLI_DATA_DIR)

if len(HOST_NAME)==0:
    from requests import get
    HOST_NAME=get('https://api.ipify.org').text
    HOST_IP=HOST_NAME
print(f" HOST_NAME is {HOST_NAME}  with IP {HOST_IP} ")


In [ ]:
HOST_NAME=HOST_IP
print(HOST_NAME)

---
# 1.0 What Is XNAT 
According to [XNAT main site](https://www.xnat.org/), 
XNAT is an open source imaging informatics platform developed by the Neuroinformatics Research Group at Washington University. 
XNAT was originally developed at Washington University in the Buckner Lab, which is now located at Harvard University. 
It facilitates common management, productivity, and quality assurance tasks for imaging and associated data. 
Thanks to its extensibility, XNAT can be used to support a wide range of imaging-based projects. 
<p align = "center"><img src="./images/xnat_intro.png" width="900"/></p>
Xnat provides the following features:
1- Full DICOM Integration and Anonymization: Get image data in, and keep PHI out
2- Secure Access & Permission Control: You decide who does what with your data.
3- Integrated Search & Reporting: Report on your image and clinical data together.
4- Pipeline Processing: Use the power of high-performance computing on your data.
5- Modular Extensibility: Expand the capabilities of your XNAT to meet your needs.
6- Developer Community: Benefit from an active and engaged set of XNAT power users.

 
## 1.1 Get to the XNAT Site
We have set up XNAT with the needed plugins for you in this lab. 
It should be already up and running.  
Run the cell below to get link to the site.

<div class="alert alert-block alert-info">
Username: <b>admin</b><br>
Password: <b>admin</b>
</div>

In [ ]:
url_text='<h1><center><a href="http://'+HOST_NAME+':7781" target="_blank">'
url_text=url_text+'<img src="https://www.xnat.org/images/XNAT-logo-980.png" width="200" height="100"/>'
url_text=url_text+'<br>Click to open xnat <br> in a new window</a></center></h1>'
# print (url_text)
ipywidgets.HTML(url_text)

The landing page of XNAT should be similar to image below.

<p align = "center"><img src="./images/xnat_login.png" alt="drawing" width="900"/></p>

Next on the site setup page (as image below), simply scroll all the way to the bottom and click save leaving all default values.
<p align = "center"><img src="./images/xnat_setup_top.png" alt="drawing" width="500"/></p>
<p align = "center"><img src="./images/xnat_setup_bot.png" alt="drawing" width="500"/></p>

<div class="alert alert-danger"> 
DO NOT continue to next step until you have configured the XNAT site by accepting the defaults as shown above. 

Failing to complete this step first may result in a broken XNAT setup.
</div>

## 1.2 Configure XNAT
This step will:
- Set up site welcome page and set some login parameters
- Configure XNAT DICOM receiver
- Create a sample project

In [ ]:
!/dli/scripts/xnat_setup.sh configure_xnat

## 1.3 Confirm XNAT Is Configured Correctly
- To confirm the site welcome page, you can simply log out then log back in.
- Once you are logged back in, you should see one project called `project1`
- Confirm DICOM receiver was set up correctly with a rule, which makes any DICOM sent to that receiver go directly to `project1`.  Do this by running the cell below.

In [ ]:
# from IPython.display import display, HTML
import base64

with open('/dli/notebooks/images/xnat_dcm_reciever.png', 'rb') as f:
    image_data = f.read()
image_base64 = base64.b64encode(image_data).decode('utf-8')
url_text='<h1><center><a href="http://'+HOST_NAME+':7781/app/template/Page.vm?view=admin#tab=dicom-scp-receivers-tab" target="_blank">'
url_text = url_text + f'<img src="data:image/jpeg;base64,{image_base64}" width="800" alt="Local Image">'
# Create and display the HTML widget
ipywidgets.HTML(value=url_text)

In case clicking the image above doesn't work, you can get to it from 
XNAT toolbar --> Administer --> site administration, 
from  the left hand side menu select Dicom SCP receiver. 
Click edit to DICOM receiver with routing rule as 
```
(0008,0018):(.*):1 t:.+ r:project1
``` 
This rule makes any DICOM sent to that receiver go directly to project1. 
    You can make multiple receivers on different ports so you can direct images to different projects,  
    then run different AI models as we will see in the next notebook.  
    

---
# 2.0 Prepare DICOM Sample Data

## 2.1 Download Spleen CT Data
You can skip this step as we have already downloaded this dataset for you in this DLI. 

## 2.2 Convert to DICOM Format
You can skip this step if you already used the OHIF notebook.
To use the DICOM server, the NIFTI format data needs to be converted to DICOM format.  
`Plastimatch` is an open tool to generate simple DICOM data from NIFTI file. 
The following commands show how to convert sample data.

```python

##### convert one of the NIFTI images to DICOM: name: <patient1>, output folder: <dicom_output>
!mkdir -p DICOMs
for i in ["1","11","15","23","30"]:
    print(f"converting spleen {i}")
    inputPath="datasets/Task09_Spleen/imagesTs/spleen_"+i+".nii.gz"
    outputPath="DICOMs/spleen_"+i
    patientID="patient"+i
    !plastimatch convert --study-description $patientID --patient-name $patientID --patient-id $patientID --input $inputPath --output-dicom $outputPath

```

In [ ]:
!ls

## 2.3 Upload to XNAT
Here we will push only two DICOM patients to XNAT PACS. 
We will use the rest in the next notebook.

In [ ]:
!python -m pynetdicom storescu dli-xnat-web 8104 DICOMs/spleen_1 -aet Project1 -aec XNAT -r
!python -m pynetdicom storescu dli-xnat-web 8104 DICOMs/spleen_15 -aet Project1 -aec XNAT -r
!python -m pynetdicom storescu dli-xnat-web 8104 DICOMs/spleen_23 -aet Project1 -aec XNAT -r

## 2.4 Convert Liver Dataset

In [ ]:
!python -m pynetdicom storescu dli-xnat-web 8104 DICOMs/liver* -aet Project1 -aec XNAT -r

---
# 3.0 Set Up XNAT OHIF
Here we need to set the address of MONAI Label server in the XNAT OHIF plugin.

In [ ]:
USER_PASS="admin:admin"
AIAASERVERPORT="http://"+HOST_NAME+":8000"
ROOTURL="http://dli-xnat-web:8080"
!curl -u $USER_PASS -X PUT "$ROOTURL/xapi/ohifaiaa/servers" \
        -H "accept: */*" -H "Content-Type: application/json" \
        -d "[ \"$AIAASERVERPORT\"]"

---
# 4.0 Start MONAI Label Server
The command will start MONAI Label server with all available models. 
If the user wants to start with a specific model, 
set the model name in the **--conf models** argument.

In [ ]:
# export MONAI_LABEL_DATASTORE=xnat
%env MONAI_LABEL_DATASTORE=xnat
# export MONAI_LABEL_DATASTORE_USERNAME=admin
%env MONAI_LABEL_DATASTORE_USERNAME=admin
# export MONAI_LABEL_DATASTORE_PASSWORD=admin
%env MONAI_LABEL_DATASTORE_PASSWORD=admin

# export MONAI_LABEL_DATASTORE_ASSET_PATH=$XNAT_DATA_ARCHIVE
%env MONAI_LABEL_DATASTORE_ASSET_PATH=$XNAT_DATA_ARCHIVE

# export MONAI_LABEL_DATASTORE_PROJECT=
# export MONAI_LABEL_DATASTORE_CACHE_PATH=
#!export MONAI_LABEL_DICOMWEB_CONVERT_TO_NIFTI=false
%env MONAI_LABEL_DICOMWEB_CONVERT_TO_NIFTI=false

os.chdir(DLI_DATA_DIR)
!monailabel start_server \
    --app apps/radiology \
    --studies http://xnat-web:8080 \
    --conf models segmentation_spleen

---
# 5.0 Auto-Segmentation with XNAT OHIF

## 5.1 MONAI Label Plugin

Go to xnat then go to 
1. Click project1
2. Select any subject 
3. Click "view Images" on the righthand side 
<p align = "center"><img src="./images/xnat_viewimage.png" alt="drawing" width="700"/></p>
4. From the right hand side, click on options --> preferences
<p align = "center"><img src="./images/xnat_ohif_preference.png" alt="drawing" width="400"/></p>
5. Select MONAI Label from the experimental tab
<p align = "center"><img src="./images/xnat_ohif_experimental.png" alt="drawing" width="400"/></p>
6. After the step above, a new button `monai label` should appear in the `mask` toolbar
<p align = "center"><img src="./images/xnat_ohif_select_ml.png" alt="drawing" width="900"/></p>


## 5.2 Run Auto-Segmentation
Similar to OHIF, you will connect to MONAI Label server and see all the models.
You should select a model and run inference to trigger a call to MONAI Label to run inference.
After couple of minutes, you will see the segmentation result in the OHIF viewer. 


<img src="images/nvidia_header.png" style="margin-left: -30px; width: 300px; float: left;">